In [ ]:
!pip install streamlit -q

In [ ]:
!pip install pyngrok -q

In [ ]:
from pyngrok import ngrok, conf

In [ ]:
#set authtoken here:
ngrok.set_auth_token('2aV65F6C7qdTfXJBwPK1ypj8eUm_7pAYbuScJjEX7paVYSotC')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd "/content/drive/MyDrive/Hunch_Models"

/content/drive/MyDrive/Hunch_Models


In [ ]:
#streamlit application
%%writefile fer.py

import os
import tensorflow as tf
import pandas as pd
import streamlit as st
import numpy as np
import cv2
from PIL import Image
import pickle
import datetime

# title

# st.markdown("<h1 style='text-align: center; color: White; background-color:#4287f5'>HR.DAYA</h1>", unsafe_allow_html=True)
# st.markdown("<h4 style='text-align: center; color: White; background-color:#de3c4b'> This app requires access to your camera to capture images.</h1>", unsafe_allow_html=True)

st.markdown("<h1 style= 'text-align: center; background: #30638E;'>Welcome to HR.Daya!</h1>", unsafe_allow_html=True)
st.markdown("<h2 style= 'text-align: center; background: #EDAE49;'>Can a Machine Tell What You're Feeling?</h2>", unsafe_allow_html=True)

st.markdown("<h6 style= 'text-align: center;'>This app requires access to your camera to capture images!</h6>", unsafe_allow_html=True)

# sidebar
st.markdown("""
<style>
    [data-testid=stSidebar] {
        background-color: #30638E;
    }
</style>
""", unsafe_allow_html=True)


st.sidebar.header("What is HR.DAYA?")
st.sidebar.text("HR.DAYA is named after the Sanskrit word hṛdaya which means heart or mind. HR.DAYA is a Facial Emotion Recognition (FER) AI model that determines a subject's emotion.")
st.sidebar.header("What tools where used to make this?")
st.sidebar.text("HR.DAYA was trained on the Chicago Face Database (CFD), programmed in Google Colab. HR.DAYA was made using Keras and Tensorflow. And finally, Streamlit turns data scripts into shareable web apps.")
st.sidebar.markdown("<h1 style= 'background: #EDAE49;'>Legend:</h1>", unsafe_allow_html = True )
st.sidebar.markdown("N: Neutral\nH: Happy\nF: Frustrated\nA: Angry")

# functions
model = pickle.load(open('/content/drive/MyDrive/Hunch_Models/fermodel.pkl', 'rb'))
# with open('/content/drive/MyDrive/fermodel.pkl', 'wb') as f:
#   pickle.dump(model, f)

# with st.spinner('Loading Model...'):
#     model = load_model()

# categories = ['H', 'N', 'A', 'F'] # Happy, Neutral, Angry, Frustrated

# taking the picture

image = st.camera_input('Photo Booth', help = 'This picture will be fed into an AI model to predict your emotion!')

# processing the picture
if image:
  image_pil = Image.open(image)

  # convert PIL Image to NumPy array
  image_np = np.array(image_pil)

  # resize img to match model's input shape
  image_np = cv2.resize(image_np, (300, 200))
  image_np = image_np.reshape(1, 300, 200, 3)

  # normalize image data
  image_np = image_np / 255.0

  # uses image_np for prediction

  categories = ['H', 'N', 'A', 'F']

  prediction = model.predict(image_np)
  predicted_class_index = np.argmax(prediction)
  predicted_category = categories[predicted_class_index]
  st.success(f"Predicted Emotion: {predicted_category}")

# Feedback

def save_feedback_to_file(image_array, predicted_category, correct, user_label=None):
    import os
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

    image_pil = Image.open(image)
    image_np = np.array(image_pil)

    image_bgr = cv2.cvtColor(image_np, cv2.COLOR_RGB2BGR)

    os.makedirs("feedback", exist_ok=True)

    # create the filename
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename_parts = [timestamp, predicted_category, correct]
    if user_label:
        filename_parts.append(f"user_{user_label}")
    filename = "_".join(filename_parts) + ".jpg"

    # save the image
    filepath = os.path.join("feedback", filename)
    cv2.imwrite(filepath, image_bgr)

# UI

st.markdown("<h4 style= 'text-align: center; background: #D1495B'> Provide Feedback! </h4>", unsafe_allow_html=True)
st.markdown("<h6 style= 'text-align: center; background: #00798C'> Was this prediction correct? </h6>", unsafe_allow_html=True)

# Radio button component without extra space
correct = st.radio("", ["Yes", "No"], horizontal=True)

user_label = None
if correct == "No":
    user_label = st.text_input("Please provide the correct emotion (N, H, F, A):")

if st.button("Submit Feedback"):
    if correct == "Yes":
        save_feedback_to_file(image_np, predicted_category, "Yes")
        st.success("Thanks for confirming! 😊")
    elif user_label.upper() in categories:
        save_feedback_to_file(image_np, predicted_category, "No", user_label.upper())
        st.success("Thank you for your feedback! We'll use this to improve HR.DAYA.")
    else:
        st.error("Invalid input. Please enter only N, H, F, or A.")


Overwriting fer.py


In [ ]:
# pyngrok url to view streamlit application!
# this works on the hotspot!

!pkill -f "streamlit"
!pkill -f "ngrok"

!nohup streamlit run fer.py --server.port 8501 &
from pyngrok import ngrok

url = ngrok.connect(addr = '8501', proto="http", bind_tls=True)
print(url)

nohup: appending output to 'nohup.out'
NgrokTunnel: "https://6df0-34-59-9-245.ngrok-free.app" -> "http://localhost:8501"
